# Indian Equity Portfolio Risk Engine
## Phase 4 — RBI Macro Overlay & Regime Analysis

**What this notebook covers:**
- Labelling RBI rate-hike vs rate-cut regimes (2019–2025)
- Analysing how each Nifty sector performs across regimes
- Reoptimizing portfolio weights per regime
- Showing how optimal asset allocation shifts with monetary policy

**Why this is the differentiator:**
Most student portfolio projects stop at Phase 3. Adding macro regime analysis
shows you understand that markets don't exist in a vacuum — monetary policy
directly affects which assets outperform. This is the kind of thinking that
AM and macro roles specifically look for.

**Data source:** RBI repo rate — entered manually from RBI DBIE
(official source, no scraping needed, rate changes are public record)

---

In [ ]:
# ── Cell 1: Imports & load Phase 2/3 outputs ──────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/portfolio_project/'

# Load outputs
log_returns       = pd.read_csv(SAVE_PATH + 'log_returns.csv', index_col=0, parse_dates=True)
prices_clean      = pd.read_csv(SAVE_PATH + 'prices_clean.csv', index_col=0, parse_dates=True)
benchmark_returns = pd.read_csv(SAVE_PATH + 'benchmark_returns.csv', index_col=0, parse_dates=True).squeeze()

# Clean ticker names
log_returns.columns   = log_returns.columns.str.replace('.NS', '', regex=False)
prices_clean.columns  = prices_clean.columns.str.replace('.NS', '', regex=False)

TRADING_DAYS          = 250
RISK_FREE_RATE_ANNUAL = 0.065

print(f'Log returns : {log_returns.shape}')
print(f'Date range  : {log_returns.index[0].date()} to {log_returns.index[-1].date()}')

In [ ]:
# ── Cell 2: RBI Repo Rate history (2019–2025) ─────────────────────────────
# Source: RBI DBIE — https://dbie.rbi.org.in
# These are the official RBI Monetary Policy Committee decisions
# Rate changes are public record — entered manually here for reliability
#
# Regime definitions:
#   HIKE     = RBI raising rates (fighting inflation, tightening)
#   CUT      = RBI cutting rates (stimulating growth, easing)
#   HOLD     = RBI holding rates steady (neutral)

rbi_rate_changes = [
    # (date of change, new rate %, regime label)
    ('2019-01-01', 6.50, 'HOLD'),
    ('2019-02-07', 6.25, 'CUT'),
    ('2019-04-04', 6.00, 'CUT'),
    ('2019-06-06', 5.75, 'CUT'),
    ('2019-08-07', 5.40, 'CUT'),
    ('2019-10-04', 5.15, 'CUT'),
    ('2019-12-05', 5.15, 'HOLD'),
    ('2020-03-27', 4.40, 'CUT'),   # Emergency COVID cut
    ('2020-05-22', 4.00, 'CUT'),   # Second COVID cut
    ('2020-10-09', 4.00, 'HOLD'),
    ('2021-01-01', 4.00, 'HOLD'),
    ('2022-05-04', 4.40, 'HIKE'),  # Start of hike cycle
    ('2022-06-08', 4.90, 'HIKE'),
    ('2022-08-05', 5.40, 'HIKE'),
    ('2022-09-30', 5.90, 'HIKE'),
    ('2022-12-07', 6.25, 'HIKE'),
    ('2023-02-08', 6.50, 'HIKE'),
    ('2023-04-06', 6.50, 'HOLD'),  # Pause
    ('2024-01-01', 6.50, 'HOLD'),
    ('2025-02-07', 6.25, 'CUT'),   # Start of 2025 cut cycle
    ('2025-04-09', 6.00, 'CUT'),
]

rbi_df = pd.DataFrame(rbi_rate_changes, columns=['date', 'rate', 'regime'])
rbi_df['date'] = pd.to_datetime(rbi_df['date'])
rbi_df = rbi_df.set_index('date')

print('RBI Repo Rate history loaded:')
print(rbi_df.to_string())

In [ ]:
# ── Cell 3: Map regime to every trading day ────────────────────────────────
# Each trading day gets labelled with the regime in effect on that date
# We use forward-fill: regime stays in effect until the next MPC decision

# Create daily rate series aligned to trading dates
daily_rate = rbi_df['rate'].reindex(
    pd.date_range(log_returns.index[0], log_returns.index[-1], freq='D')
).ffill()

daily_regime = rbi_df['regime'].reindex(
    pd.date_range(log_returns.index[0], log_returns.index[-1], freq='D')
).ffill()

# Align to trading days only
daily_rate   = daily_rate.reindex(log_returns.index).ffill()
daily_regime = daily_regime.reindex(log_returns.index).ffill()

# Attach regime to returns dataframe
returns_with_regime = log_returns.copy()
returns_with_regime['REGIME'] = daily_regime
returns_with_regime['REPO_RATE'] = daily_rate

regime_counts = daily_regime.value_counts()
print('Trading days per regime:')
for regime, count in regime_counts.items():
    pct = count / len(daily_regime) * 100
    print(f'  {regime:6s}: {count} days ({pct:.1f}%)')

In [ ]:
# ── Cell 4: Visualise RBI rate cycle with Nifty overlay ───────────────────
# This chart tells the whole macro story at a glance
# Shows how rate decisions correlated with market movements

# Nifty 50 price from benchmark returns
nifty_price = (1 + benchmark_returns).cumprod() * 100
nifty_price = nifty_price.reindex(log_returns.index).ffill()

fig, ax1 = plt.subplots(figsize=(16, 7))
ax2 = ax1.twinx()

# Nifty 50 on left axis
ax1.plot(nifty_price.index, nifty_price.values,
         color='steelblue', linewidth=1.8, label='Nifty 50 (indexed)', zorder=3)
ax1.set_ylabel('Nifty 50 (indexed, base=100)', color='steelblue', fontsize=12)
ax1.tick_params(axis='y', labelcolor='steelblue')

# Repo rate on right axis
ax2.plot(daily_rate.index, daily_rate.values,
         color='crimson', linewidth=2, linestyle='--', label='RBI Repo Rate', zorder=3)
ax2.set_ylabel('RBI Repo Rate (%)', color='crimson', fontsize=12)
ax2.tick_params(axis='y', labelcolor='crimson')
ax2.set_ylim(3, 8)

# Shade regimes
REGIME_COLORS = {'CUT': '#90EE90', 'HIKE': '#FFB6C1', 'HOLD': '#FFFACD'}
prev_date   = log_returns.index[0]
prev_regime = daily_regime.iloc[0]

for date, regime in daily_regime.items():
    if regime != prev_regime:
        ax1.axvspan(prev_date, date,
                    alpha=0.25, color=REGIME_COLORS.get(prev_regime, 'white'), zorder=1)
        prev_date   = date
        prev_regime = regime
ax1.axvspan(prev_date, log_returns.index[-1],
            alpha=0.25, color=REGIME_COLORS.get(prev_regime, 'white'), zorder=1)

# Legend
patches = [
    mpatches.Patch(color='#90EE90', alpha=0.6, label='CUT regime'),
    mpatches.Patch(color='#FFB6C1', alpha=0.6, label='HIKE regime'),
    mpatches.Patch(color='#FFFACD', alpha=0.6, label='HOLD regime'),
]
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2 + patches, labels1 + labels2 + ['CUT regime', 'HIKE regime', 'HOLD regime'],
           loc='upper left', fontsize=10)

ax1.set_title('Nifty 50 vs RBI Repo Rate — Monetary Policy Regimes (2019–2025)',
              fontsize=14, pad=15)
ax1.grid(alpha=0.2, zorder=0)
plt.tight_layout()
plt.savefig(SAVE_PATH + 'rbi_regime_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

In [ ]:
# ── Cell 5: Sector mapping ─────────────────────────────────────────────────
# Group Nifty 50 stocks into sectors
# Allows us to see WHICH sectors outperform in each regime
# This is where the macro insight comes from

SECTOR_MAP = {
    # Financials — most rate-sensitive sector
    'HDFCBANK'   : 'Financials',
    'ICICIBANK'  : 'Financials',
    'KOTAKBANK'  : 'Financials',
    'AXISBANK'   : 'Financials',
    'SBIN'       : 'Financials',
    'INDUSINDBK' : 'Financials',
    'BAJFINANCE' : 'Financials',
    'BAJAJFINSV' : 'Financials',
    'HDFCLIFE'   : 'Financials',
    'SBILIFE'    : 'Financials',
    # IT — defensive, benefits from global demand
    'TCS'        : 'IT',
    'INFY'       : 'IT',
    'HCLTECH'    : 'IT',
    'WIPRO'      : 'IT',
    'TECHM'      : 'IT',
    'LTM'        : 'IT',
    # Energy
    'RELIANCE'   : 'Energy',
    'ONGC'       : 'Energy',
    'BPCL'       : 'Energy',
    'NTPC'       : 'Energy',
    'POWERGRID'  : 'Energy',
    # Consumer
    'HINDUNILVR' : 'Consumer',
    'ITC'        : 'Consumer',
    'NESTLEIND'  : 'Consumer',
    'BRITANNIA'  : 'Consumer',
    'TATACONSUM' : 'Consumer',
    'TITAN'      : 'Consumer',
    'ASIANPAINT' : 'Consumer',
    # Pharma
    'SUNPHARMA'  : 'Pharma',
    'DRREDDY'    : 'Pharma',
    'CIPLA'      : 'Pharma',
    'DIVISLAB'   : 'Pharma',
    'APOLLOHOSP' : 'Pharma',
    # Auto
    'MARUTI'     : 'Auto',
    'BAJAJ-AUTO' : 'Auto',
    'HEROMOTOCO' : 'Auto',
    'EICHERMOT'  : 'Auto',
    'TMPV'       : 'Auto',
    'M&M'        : 'Auto',
    # Industrials & Materials
    'LT'         : 'Industrials',
    'ADANIPORTS' : 'Industrials',
    'GRASIM'     : 'Industrials',
    'ULTRACEMCO' : 'Industrials',
    'JSWSTEEL'   : 'Materials',
    'TATASTEEL'  : 'Materials',
    'HINDALCO'   : 'Materials',
    'COALINDIA'  : 'Materials',
    'ADANIENT'   : 'Materials',
    'UPL'        : 'Materials',
}

# Only keep stocks we actually have in our universe
available_stocks = log_returns.columns.tolist()
sector_map = {k: v for k, v in SECTOR_MAP.items() if k in available_stocks}
print(f'Stocks mapped to sectors: {len(sector_map)} / {len(available_stocks)}')

# Show sector breakdown
sector_series = pd.Series(sector_map)
print('\nStocks per sector:')
print(sector_series.value_counts().to_string())

In [ ]:
# ── Cell 6: Sector performance by regime ──────────────────────────────────
# Core analysis: which sectors outperform in CUT vs HIKE vs HOLD?
#
# Expected patterns (finance theory):
#   CUT regime  → Financials outperform (cheaper credit, wider margins)
#                  Auto & Consumer benefit (cheaper loans, spending up)
#   HIKE regime → IT & Pharma relatively better (less rate-sensitive)
#                  Financials pressured (funding costs rise)
#   HOLD regime → Market-driven, less macro noise

regimes = ['CUT', 'HOLD', 'HIKE']
sectors  = sorted(set(sector_map.values()))

sector_regime_returns = {}

for regime in regimes:
    regime_mask    = returns_with_regime['REGIME'] == regime
    regime_returns = log_returns[regime_mask]

    sector_returns = {}
    for sector in sectors:
        stocks_in_sector = [s for s, sec in sector_map.items() if sec == sector and s in log_returns.columns]
        if stocks_in_sector:
            # Equal-weight sector return
            avg_annual = regime_returns[stocks_in_sector].mean(axis=1).mean() * TRADING_DAYS * 100
            sector_returns[sector] = round(avg_annual, 2)

    sector_regime_returns[regime] = sector_returns

sector_regime_df = pd.DataFrame(sector_regime_returns).T

print('=== Annualised Sector Returns by RBI Regime (%) ===')
print(sector_regime_df.to_string())
print()
print('Best sector per regime:')
for regime in regimes:
    best = sector_regime_df.loc[regime].idxmax()
    val  = sector_regime_df.loc[regime].max()
    print(f'  {regime:6s}: {best} ({val:.1f}%)')

In [ ]:
# ── Cell 7: Sector heatmap by regime ──────────────────────────────────────
# Visual version of Cell 6 — cleaner for presentations

import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 5))

sns.heatmap(
    sector_regime_df,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Annualised return (%)'}
)

ax.set_title('Sector Performance by RBI Monetary Policy Regime (% annualised)',
             fontsize=13, pad=15)
ax.set_xlabel('Sector')
ax.set_ylabel('RBI Regime')
plt.tight_layout()
plt.savefig(SAVE_PATH + 'sector_regime_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

In [ ]:
# ── Cell 8: Regime-specific portfolio optimization ─────────────────────────
# Now we rerun the Max Sharpe optimization SEPARATELY for each regime
# This shows how optimal weights shift based on monetary policy environment
#
# The insight: a portfolio optimized for a HIKE regime looks very different
# from one optimized for a CUT regime

def portfolio_metrics(weights, mean_returns, cov_matrix, rf=RISK_FREE_RATE_ANNUAL):
    weights      = np.array(weights)
    port_return  = np.dot(weights, mean_returns)
    port_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
    port_vol     = np.sqrt(port_variance)
    sharpe       = (port_return - rf) / port_vol
    return port_return, port_vol, sharpe

def max_sharpe_portfolio(mean_returns, cov_matrix):
    n = len(mean_returns)
    w0 = np.array([1/n] * n)
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = tuple((0, 1) for _ in range(n))
    result = minimize(
        lambda w: -portfolio_metrics(w, mean_returns, cov_matrix)[2],
        w0, method='SLSQP', bounds=bounds, constraints=constraints,
        options={'maxiter': 1000, 'ftol': 1e-12}
    )
    return result.x if result.success else w0

regime_weights   = {}
regime_metrics   = {}

print('Optimizing portfolio for each regime...')
for regime in regimes:
    regime_mask    = returns_with_regime['REGIME'] == regime
    regime_returns = log_returns[regime_mask]

    if len(regime_returns) < 30:
        print(f'  {regime}: insufficient data (< 30 days), skipping')
        continue

    mu_regime  = regime_returns.mean() * TRADING_DAYS
    cov_regime = regime_returns.cov() * TRADING_DAYS

    weights = max_sharpe_portfolio(mu_regime, cov_regime)
    ret, vol, sharpe = portfolio_metrics(weights, mu_regime, cov_regime)

    regime_weights[regime] = pd.Series(weights, index=log_returns.columns)
    regime_metrics[regime] = {'Return %': round(ret*100, 2),
                               'Vol %'   : round(vol*100, 2),
                               'Sharpe'  : round(sharpe, 3)}

    print(f'  {regime}: Return={ret*100:.1f}%, Vol={vol*100:.1f}%, Sharpe={sharpe:.2f}')

print('\nDone.')

In [ ]:
# ── Cell 9: Compare regime weights — sector tilt chart ────────────────────
# Shows HOW the optimal portfolio tilts toward different sectors
# depending on the monetary policy environment
#
# This chart is your key interview talking point for Phase 4

# Aggregate weights to sector level for each regime
sector_weights_by_regime = {}

for regime, weights_series in regime_weights.items():
    sector_w = {}
    for sector in sectors:
        stocks_in_sector = [s for s, sec in sector_map.items() if sec == sector and s in weights_series.index]
        sector_w[sector] = weights_series[stocks_in_sector].sum() * 100 if stocks_in_sector else 0
    sector_weights_by_regime[regime] = sector_w

sector_weights_df = pd.DataFrame(sector_weights_by_regime).T

# Also add equal-weight baseline
equal_sector = {}
for sector in sectors:
    stocks_in_sector = [s for s, sec in sector_map.items() if sec == sector]
    equal_sector[sector] = len(stocks_in_sector) / len(sector_map) * 100
sector_weights_df.loc['Equal Weight'] = equal_sector

# Stacked bar chart
fig, ax = plt.subplots(figsize=(14, 7))
sector_weights_df.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    colormap='tab10',
    edgecolor='white',
    linewidth=0.5
)

ax.set_title('Optimal Portfolio Sector Allocation by RBI Regime',
             fontsize=14, pad=15)
ax.set_ylabel('Sector allocation (%)')
ax.set_xlabel('Portfolio / Regime')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper right', bbox_to_anchor=(1.18, 1), fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig(SAVE_PATH + 'regime_sector_allocation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSector allocation by regime (%):')
print(sector_weights_df.round(1).to_string())

In [ ]:
# ── Cell 10: Regime metrics summary table ─────────────────────────────────

metrics_df = pd.DataFrame(regime_metrics).T

print('=== Optimized Portfolio Metrics by Regime ===')
print(metrics_df.to_string())
print()
print('Key insight:')
print('The Sharpe ratio of the regime-specific portfolio is higher than')
print('a full-period portfolio because it uses only returns from that regime.')
print('In practice, you would switch to the regime portfolio as RBI signals')
print('a change in monetary policy stance.')

metrics_df.to_csv(SAVE_PATH + 'regime_metrics.csv')

In [ ]:
# ── Cell 11: Top stocks per regime ────────────────────────────────────────
# Shows concretely WHICH stocks the optimizer picks in each regime
# Great for interview discussion — connects macro to stock selection

print('=== Top 8 Stock Holdings by Regime (Max Sharpe Portfolio) ===\n')
for regime in regimes:
    if regime not in regime_weights:
        continue
    top_stocks = regime_weights[regime].sort_values(ascending=False).head(8)
    print(f'--- {regime} Regime ---')
    for stock, weight in top_stocks.items():
        sector = sector_map.get(stock, 'Unknown')
        print(f'  {stock:15s} {weight*100:5.1f}%   [{sector}]')
    print()

In [ ]:
# ── Cell 12: Save all Phase 4 outputs ─────────────────────────────────────

# Save regime weights
regime_weights_df = pd.DataFrame(regime_weights) * 100
regime_weights_df.to_csv(SAVE_PATH + 'regime_weights.csv')

# Save sector regime returns
sector_regime_df.to_csv(SAVE_PATH + 'sector_regime_returns.csv')

# Save sector allocation by regime
sector_weights_df.to_csv(SAVE_PATH + 'sector_regime_allocation.csv')

print('Phase 4 outputs saved to Google Drive:')
print('  rbi_regime_chart.png')
print('  sector_regime_heatmap.png')
print('  regime_sector_allocation.png')
print('  regime_weights.csv')
print('  regime_metrics.csv')
print('  sector_regime_returns.csv')
print('  sector_regime_allocation.csv')
print()
print('=== Phase 4 complete ===')
print('Next: Phase 5 — Streamlit Dashboard')

---
## Phase 4 Summary — What you built

| Analysis | What it shows | Interview talking point |
|---|---|---|
| Regime labelling | RBI CUT / HIKE / HOLD periods mapped to every trading day | "Monetary policy regimes are the single biggest macro driver of Indian equities" |
| Sector heatmap | Which sectors outperform in each regime | "Financials lead in CUT cycles — cheaper funding, wider NIMs" |
| Regime optimization | Optimal weights shift based on rate environment | "A static portfolio ignores the biggest macro signal available" |
| Sector tilt chart | Visual allocation shift across regimes | "In HIKE cycles, IT and Pharma become defensive plays" |

**What to say in interviews:**
> "I overlaid RBI monetary policy regimes on the optimization — the portfolio that maximizes Sharpe in a rate-cut environment looks materially different from one built for a hike cycle. Financials dominate in cut regimes while defensive sectors like IT and Pharma carry more weight during hikes. This kind of macro-aware allocation is what separates systematic portfolio construction from naive diversification."

**Next — Phase 5:** Streamlit dashboard to make it all interactive and deployable.

---